# Lasso Regression (L1)

Linear regression with **L1 (lasso) regularization**. Features are standardized inside a pipeline (scaler fit on training data only, then applied to test — no leakage). `LASSO_ALPHA` controls shrinkage and, unlike ridge, **sparsity**: L1 drives some coefficients exactly to zero, so lasso also does feature selection.

Same pipeline as the ridge/OLS notebooks — one `.parquet` read once and sliced by year, the trailing/cumulative window toggle, and a 4-column output (`permno, eom, target_w, prediction`) — with the model swapped to `Lasso`.

**alpha is on a completely different scale from ridge.** sklearn's Lasso minimizes `(1/2n)||y - Xb||^2 + alpha*||b||_1`, so a ridge-style alpha of 10 would zero every coefficient. Sparsity is very sensitive to alpha; `n_selected` (nonzero coefficients) prints per fold, and you will likely want to tune alpha (LassoCV, or on the valid block the way PCR tunes M) rather than trust a fixed guess.

**Coefficient statistics are saved** to `LASSO_coef_stats.parquet`, same schema as `OLS_coef_stats.parquet` (plus `alpha`, `n_selected`), so it drops into the accuracy/importance notebook with `MODEL="LASSO"`. Deselected features have `coef` exactly 0 and `SE/t/p/CI` left NaN — the blanks are the selection pattern.

> **The SE/t/p are not valid inference.** Lasso is biased *and* it selects, so there is no closed-form sampling covariance and the usual t-tests do not apply. For the selected (nonzero) features the table reports **naive post-selection OLS** SEs — `sigma^2 (Z_S'Z_S)^-1` on the active set S, `sigma^2` using `df ~ n_selected` — which **ignore the selection event** and are anti-conservative. Read them as a rough scale signal on the chosen features, not significance tests; for valid inference use the debiased lasso or the bootstrap. The trustworthy lasso output is the **selection pattern and coefficient signs**, not the p-values.

In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from numpy.linalg import pinv
from scipy.stats import t as t_dist

In [14]:
# ---- configuration ---------------------------------------------------------
PARQUET_PATH = "US_GFD_FEATURES.parquet"
OUTPUT_PATH  = "LASSO_predictions.parquet"
MODEL_DIR    = "LASSO_models"   # fitted models saved here for feature importance
COEF_STATS_PATH = "LASSO_coef_stats.parquet"   # per-model coef / SE / t / p / CI table
CONF            = 0.95                         # confidence level for the stored CIs

FIRST_YEAR = 1984
LAST_YEAR  = 2025
N_TRAIN    = 5
N_VALID    = 5
N_TEST     = 1

# "trailing" -> fixed N_TRAIN+N_VALID-year train window sliding forward
# "cumulative" -> expanding train from FIRST_YEAR up to the test year
SCHEME = "trailing"    # or "cumulative"

# ---- Lasso (L1) params ----
LASSO_ALPHA    = 1e-4      # L1 penalty; sparsity knob
LASSO_MAX_ITER = 10000     # raise if you see a ConvergenceWarning

ID_COLS = ["permno","eom","gvkey","iid","cusip","tic","tpci","exchg",
           "shrcd","exchcd","sic","naics","trade_eom","acc_eom",
           "acc_datadate","rdq","date_buy_t1","date_sell_t1","ret_1m",
           "target","prc","prc_buy_t1","prc_sell_t1","n_days",
           "acc_age_m","target_w"]
TARGET = "target_w"

In [3]:
# ---- resolve feature list from schema (no data read) -----------------------
schema = pq.read_schema(PARQUET_PATH)
FEATURES = [c for c in schema.names if c not in ID_COLS]
READ_COLS = ["permno", "gvkey", "eom", TARGET] + FEATURES
print(f"{len(FEATURES)} feature columns")

299 feature columns


In [4]:
# ---- read once, slice by year ---------------------------------------------
import time
_t = time.time()
FULL = pq.read_table(PARQUET_PATH, columns=READ_COLS).to_pandas()
FULL["eom"] = pd.to_datetime(FULL["eom"])
FULL = FULL.astype({**{c: "float32" for c in FEATURES}, TARGET: "float32"})
FULL["_year"] = FULL["eom"].dt.year
print(f"read {len(FULL):,} rows x {len(FULL.columns)} cols in {time.time()-_t:.1f}s, "
      f"{FULL.memory_usage(deep=True).sum()/1e9:.2f} GB")

_na = FULL[FEATURES].isna().to_numpy().sum()
_inf = np.isinf(FULL[FEATURES].to_numpy()).sum()
print(f"feature NaNs: {_na:,} | Infs: {_inf:,} | target NaNs: {FULL[TARGET].isna().sum():,}")
if _na or _inf:
    FULL[FEATURES] = FULL[FEATURES].replace([np.inf, -np.inf], np.nan)
    FULL[FEATURES] = FULL[FEATURES].fillna(0.5).astype("float32")

def slice_years(y0, y1):
    sub = FULL[FULL["_year"].between(y0, y1)]
    return sub[sub[TARGET].notna()]

C:\Users\taben\AppData\Local\Temp\ipykernel_30528\1596281292.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  FULL["_year"] = FULL["eom"].dt.year


read 2,401,704 rows x 304 cols in 1642.5s, 2.93 GB
feature NaNs: 0 | Infs: 0 | target NaNs: 17,874


In [5]:
# ---- window schedule (trailing vs cumulative) ------------------------------
def make_windows(first_year, last_year, n_tr, n_va, n_te, scheme=SCHEME):
    wins = []
    test_start = first_year + n_tr + n_va
    while test_start <= last_year:
        va = (test_start - n_va, test_start - 1)
        te = (test_start, min(test_start + n_te - 1, last_year))
        if scheme == "trailing":
            tr = (test_start - n_tr - n_va, test_start - n_va - 1)
        elif scheme == "cumulative":
            tr = (first_year, test_start - n_va - 1)
        else:
            raise ValueError(f"unknown SCHEME {scheme!r}")
        wins.append((tr, va, te))
        test_start += n_te
    return wins

WINDOWS = make_windows(FIRST_YEAR, LAST_YEAR, N_TRAIN, N_VALID, N_TEST, SCHEME)
print(f"scheme={SCHEME!r}: {len(WINDOWS)} windows | first {WINDOWS[0]} | last {WINDOWS[-1]}")

scheme='trailing': 32 windows | first ((1984, 1988), (1989, 1993), (1994, 1994)) | last ((2015, 2019), (2020, 2024), (2025, 2025))


In [15]:
# ---- fit Lasso (standardize -> L1 linear) on the full train+valid block -----
def lasso_coef_stats(pipe, X, y, feature_names, test_year, alpha, conf=CONF):
    """Coefficient table for the Lasso fit. Lasso selects (exact zeros) and is a
    non-linear, biased estimator, so there is NO valid closed-form covariance. For
    the SELECTED features we report naive post-selection OLS SEs on the active set S:
    sigma^2 (Z_S'Z_S)^-1 with sigma^2 using df ~ n_selected. These IGNORE the
    selection event and are anti-conservative -- not significance tests. Deselected
    features get coef 0 and SE/t/p/CI = NaN. Nothing is refit."""
    sc  = pipe.named_steps["standardscaler"]
    las = pipe.named_steps["lasso"]
    Z   = np.asarray(sc.transform(X), np.float64)
    Zc  = Z - Z.mean(0)
    yv  = np.asarray(y, np.float64)
    coef = np.ravel(las.coef_).astype(np.float64)          # L1 -> many exact zeros
    e    = yv - pipe.predict(X)
    n, p = Zc.shape
    active = np.flatnonzero(coef != 0.0)
    k = int(active.size)
    dof = max(n - k - 1.0, 1.0)                             # df(lasso) ~ # nonzero
    sig2 = float(e @ e) / dof
    se = np.full(p, np.nan); tval = np.full(p, np.nan)
    pval = np.full(p, np.nan); cl = np.full(p, np.nan); ch = np.full(p, np.nan)
    tcr = t_dist.ppf(1 - (1 - conf) / 2, dof)
    if k > 0:
        Zs   = Zc[:, active]
        se_s = np.sqrt(np.clip(np.diag(sig2 * pinv(Zs.T @ Zs)), 0.0, None))
        se[active]   = se_s
        tval[active] = coef[active] / np.where(se_s > 0, se_s, np.nan)
        pval[active] = 2.0 * t_dist.sf(np.abs(tval[active]), dof)
        cl[active]   = coef[active] - tcr * se_s
        ch[active]   = coef[active] + tcr * se_s
    a0  = float(las.intercept_); se0 = float(np.sqrt(sig2 / n))
    t0  = a0 / se0 if se0 > 0 else np.nan
    beta   = np.concatenate([[a0], coef]);      se_all = np.concatenate([[se0], se])
    t_all  = np.concatenate([[t0], tval])
    p_all  = np.concatenate([[2.0 * t_dist.sf(abs(t0), dof)], pval])
    cl_all = np.concatenate([[a0 - tcr * se0], cl])
    ch_all = np.concatenate([[a0 + tcr * se0], ch])
    recon  = float(np.max(np.abs((a0 + Z @ coef) - pipe.predict(X))))
    df = pd.DataFrame({
        "model_year": np.int32(test_year),
        "feature":    ["intercept"] + list(feature_names),
        "coef": beta, "abs_coef": np.abs(beta), "SE": se_all,
        "t_stat": t_all, "p_value": p_all,
        "ci_low": cl_all, "ci_high": ch_all,
        "n_obs": np.int64(n), "dof": np.float64(dof),
        "alpha": np.float64(alpha), "n_selected": np.int32(k),
    })
    return df, recon


def fit_predict_window(Xtr, ytr, Xte, Xtr_only, ytr_only, Xva, yva,
                       test_year=None, model_dir=None, feature_names=None):
    # standardize features (fit on train only), then L1-penalized linear reg.
    import joblib
    m = make_pipeline(StandardScaler(), Lasso(alpha=LASSO_ALPHA, max_iter=LASSO_MAX_ITER))
    m.fit(Xtr, ytr)
    if model_dir is not None:
        joblib.dump(m, f"{model_dir}/lasso_test{test_year}.joblib")
    stats, recon = lasso_coef_stats(m, Xtr, ytr, feature_names, test_year, LASSO_ALPHA)
    return m.predict(Xte), stats, recon


In [16]:
# ---- roll through windows, write predictions + coefficient stats -----------
import json
from pathlib import Path
Path(MODEL_DIR).mkdir(exist_ok=True)
json.dump(FEATURES, open(f"{MODEL_DIR}/feature_names.json", "w"))
writer = None
coef_stats = []
recon_max = 0.0
for (tr, va, te) in WINDOWS:
    train_df = slice_years(tr[0], va[1])     # train+valid block = training data
    test_df  = slice_years(te[0], te[1])     # the held-out test year

    Xtr = train_df[FEATURES].to_numpy(np.float32)
    ytr = train_df[TARGET].to_numpy(np.float32)
    Xte = test_df[FEATURES].to_numpy(np.float32)

    va_mask = train_df["eom"].dt.year.between(*va)
    tr_mask = train_df["eom"].dt.year.between(*tr)
    Xtr_only = train_df.loc[tr_mask, FEATURES].to_numpy(np.float32)
    ytr_only = train_df.loc[tr_mask, TARGET].to_numpy(np.float32)
    Xva = train_df.loc[va_mask, FEATURES].to_numpy(np.float32)
    yva = train_df.loc[va_mask, TARGET].to_numpy(np.float32)

    yhat, stats, recon = fit_predict_window(Xtr, ytr, Xte, Xtr_only, ytr_only, Xva, yva,
                                            te[0], MODEL_DIR, FEATURES)
    coef_stats.append(stats)
    recon_max = max(recon_max, recon)

    out = test_df[["permno", "gvkey", "eom", TARGET]].copy()
    out["prediction"] = yhat.astype(np.float32)
    tbl = pa.Table.from_pandas(out, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_PATH, tbl.schema)
    writer.write_table(tbl)
    print(f"test {te[0]}: train {tr}, valid {va} | n_train={len(Xtr):,} n_test={len(Xte):,} "
          f"| n_selected={int(stats['n_selected'].iloc[0])} | pred mean={yhat.mean():.5f}")

if writer is not None:
    writer.close()
coef_df = pd.concat(coef_stats, ignore_index=True)
coef_df.to_parquet(COEF_STATS_PATH, index=False)
print(f"done -> {OUTPUT_PATH}")
_ns = coef_df.groupby("model_year")["n_selected"].first()
print(f"coef stats: {coef_df.shape[0]:,} rows across {coef_df['model_year'].nunique()} "
      f"models (alpha={LASSO_ALPHA}, n_selected {int(_ns.min())}-{int(_ns.max())} of {len(FEATURES)}) "
      f"-> {COEF_STATS_PATH}")
print(f"max |hand-computed fitted - saved prediction| = {recon_max:.2e}")

test 1994: train (1984, 1988), valid (1989, 1993) | n_train=443,269 n_test=63,282 | n_selected=141 | pred mean=0.00993
test 1995: train (1985, 1989), valid (1990, 1994) | n_train=471,653 n_test=65,910 | n_selected=139 | pred mean=0.00893
test 1996: train (1986, 1990), valid (1991, 1995) | n_train=501,250 n_test=68,176 | n_selected=140 | pred mean=0.01040
test 1997: train (1987, 1991), valid (1992, 1996) | n_train=531,284 n_test=70,228 | n_selected=145 | pred mean=0.01087
test 1998: train (1988, 1992), valid (1993, 1997) | n_train=559,786 n_test=69,936 | n_selected=144 | pred mean=0.01260
test 1999: train (1989, 1993), valid (1994, 1998) | n_train=585,773 n_test=68,248 | n_selected=143 | pred mean=0.01128
test 2000: train (1990, 1994), valid (1995, 1999) | n_train=608,917 n_test=67,130 | n_selected=144 | pred mean=0.01282
test 2001: train (1991, 1995), valid (1996, 2000) | n_train=630,106 n_test=64,380 | n_selected=156 | pred mean=0.01386
test 2002: train (1992, 1996), valid (1997, 2001

In [17]:
preds = pd.read_parquet(OUTPUT_PATH)
print(preds.shape, preds.columns.tolist())
preds.head()

(1937791, 5) ['permno', 'gvkey', 'eom', 'target_w', 'prediction']


,permno,gvkey,eom,target_w,prediction
0,10001,12994,1994-01-31,0.000000,0.002569
1,10001,12994,1994-02-28,-0.028169,0.003797
2,10001,12994,1994-03-31,-0.119403,0.004713
3,10001,12994,1994-04-30,0.000000,0.004474
4,10001,12994,1994-05-31,0.078125,0.004566


In [18]:
# ---- preview the stored coefficient statistics -----------------------------
cstats = pd.read_parquet(COEF_STATS_PATH)
print(cstats.shape, cstats.columns.tolist())
_last = cstats["model_year"].max()
_sel = cstats[(cstats["model_year"] == _last) & (cstats["coef"] != 0)]
print(f"model {_last}: {len(_sel)-1} features selected (nonzero)")
_sel.reindex(_sel["t_stat"].abs().sort_values(ascending=False).index)[
    ["feature","coef","SE","t_stat","p_value","ci_low","ci_high","n_selected"]].head(12).round(4)

(9600, 13) ['model_year', 'feature', 'coef', 'abs_coef', 'SE', 't_stat', 'p_value', 'ci_low', 'ci_high', 'n_obs', 'dof', 'alpha', 'n_selected']
model 2025: 164 features selected (nonzero)


,feature,coef,SE,t_stat,p_value,ci_low,ci_high,n_selected
9596,ami_126d,-0.0127,0.0010,-12.9938,0.0,-0.0146,-0.0108,164
9597,dolvol_126d,-0.0113,0.0009,-12.2417,0.0,-0.0131,-0.0095,164
9300,intercept,0.0022,0.0002,12.1752,0.0,0.0019,0.0026,164
9574,rvol_252d,0.0066,0.0006,11.4530,0.0,0.0054,0.0077,164
9305,assets,0.0076,0.0008,9.8655,0.0,0.0061,0.0091,164
9474,sales,-0.0057,0.0006,-8.9830,0.0,-0.0069,-0.0044,164
9589,ret_9_1,0.0026,0.0004,6.8866,0.0,0.0019,0.0034,164
9594,rmax5_rvol_21d,0.0021,0.0003,5.9668,0.0,0.0014,0.0028,164
9548,opex_at,-0.0035,0.0006,-5.6104,0.0,-0.0047,-0.0023,164
9593,ret_1_0,-0.0021,0.0004,-5.4540,0.0,-0.0029,-0.0014,164
